# Figures and tables used in the paper (cas-sc-template\_eng\_v1)

All figures and tables of the paper are drawn in this notebook from the computed
results for the three therapeutic groups (A10B antidiabetics, N03A antiepileptics,
B01A antithrombotics). Nothing is written to file.

## Conventions

**Unmet demand and substitution volumes are aggregated with the CVaR weights
(0.7 on S2, 0.3 on S3).**
With scenario probabilities $(0.5,0.35,0.15)$ and $\alpha=0.5$, the upper 50% tail is
exactly filled by S2 and S3, so $\mathrm{CVaR}=0.7f^{S2}+0.3f^{S3}$ and the
small-scale scenario S1 does not enter the objective. The second-stage allocation of
S1 is therefore not pinned down by optimality, and a plain expectation would mix in
half an allocation that was never optimised (Section 5.4 of the paper).

**Objective values cannot be compared directly across therapeutic groups.** The weights
$\omega$ and $\pi$ are proportional to each group's demand-weighted average drug price
$\bar c$ (106.79 / 274.92 / 152.50 JPY per patient-day), so the units differ. Compare
rates of change and composition shares instead.

**The baseline case is $(\kappa_p,\kappa_c)=(\infty,1)$:** the stockpile target of each
therapeutic class is held at seven days of routine consumption, while the product mix
within a class is free.

**Product and ingredient names stay in Japanese** wherever they come straight from the
source data (the Japanese drug price listing), because they have no official English
equivalent.

## Setup

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm, LinearSegmentedColormap

SQ = os.path.abspath('.')          # this notebook lives directly under SQUID/

plt.rcParams.update({
    # Latin text in DejaVu Sans; the Japanese fonts are a fallback so that the
    # product and municipality names that come straight from the source data
    # (drug price listing / N03 boundary file) still render.
    'font.family': ['DejaVu Sans', 'Hiragino Sans', 'BIZ UDGothic'],
    'figure.dpi': 110, 'savefig.dpi': 110,
    'font.size': 9, 'axes.titlesize': 9.5, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#8a8a85', 'axes.linewidth': 0.6,
    'xtick.color': '#5a5a55', 'ytick.color': '#5a5a55',
    'axes.labelcolor': '#2b2b28', 'text.color': '#2b2b28',
    'axes.grid': True, 'grid.color': '#e6e6e1', 'grid.linewidth': 0.5,
    'axes.axisbelow': True,
})
# The English column names are longer than the Japanese ones, so widen the
# text repr to stop pandas from collapsing columns into '...'.
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 50)

# Therapeutic group: code / folder / tag prefix / colour / marker / display name.
# Colours were checked for colour-vision deficiency (OKLab dE 9.2 for adjacent
# pairs, 27.6 for normal vision) and are double-encoded by marker shape so the
# figures survive black-and-white printing.
CLASSES = [
    ('A10B', 'v9',      '',      '#2a78d6', 'o', 'Antidiabetics'),
    ('N03A', 'v9_N03A', 'N03A_', '#eb6834', 's', 'Antiepileptics'),
    ('B01A', 'v9_B01A', 'B01A_', '#1baf7a', '^', 'Antithrombotics'),
]
COL  = {c: col for c, _, _, col, _, _ in CLASSES}
MK   = {c: m   for c, _, _, _, m, _ in CLASSES}
NAME = {c: n   for c, _, _, _, _, n in CLASSES}
FOLD = {c: f   for c, f, _, _, _, _ in CLASSES}
PRE  = {c: p   for c, _, p, _, _, _ in CLASSES}
INK2 = '#5a5a55'

# High-risk ingredients: on the JADM list of essential drugs for the hyperacute
# phase of a disaster, and single-ingredient products only.
HIGH = {'A10B': ['A10BA02', 'A10BH08', 'A10BK07'],
        'N03A': ['N03AA02', 'N03AX14'],
        'B01A': ['B01AA03', 'B01AC06', 'B01AF03']}

W = {'S2': 0.7, 'S3': 0.3}        # CVaR weights (S1 is outside the objective)
BASE = dict(budget_rate=0.5, alpha=0.5, p_m=0.4, r_high=5.0, delta1=0.5, delta2=0.75)

# Sequential palette (single hue, light to dark) for magnitude-encoding figures.
BLUES = LinearSegmentedColormap.from_list(
    'blues', ['#ffffff', '#cde2fb', '#9ec5f4', '#5598e7', '#2a78d6', '#184f95'])


# ───────── Saving the figures ─────────
# The file names are identical to those in figs2_1/, so switching the folder name
# in the .tex file is all that is needed to use the English figures.
FIGDIR = os.path.abspath(os.path.join(SQ, '..', '..', 'Tex_薬剤配送', 'figs2_1_eng'))
SAVE = True          # set to False to display only, without writing files


def savefig(fig, *names):
    if not SAVE or not names:
        return
    os.makedirs(FIGDIR, exist_ok=True)
    for n in names:
        fig.savefig(os.path.join(FIGDIR, n), bbox_inches='tight',
                    dpi=200 if n.endswith('.png') else None)
    print('saved -> %s' % ', '.join(names))

# Romanised municipality names, keyed by the city-level code used on the node side.
MUNI_EN = {
    22100: 'Shizuoka',   22130: 'Hamamatsu',  22203: 'Numazu',     22205: 'Atami',
    22206: 'Mishima',    22207: 'Fujinomiya', 22208: 'Ito',        22209: 'Shimada',
    22210: 'Fuji',       22211: 'Iwata',      22212: 'Yaizu',      22213: 'Kakegawa',
    22214: 'Fujieda',    22215: 'Gotemba',    22216: 'Fukuroi',    22219: 'Shimoda',
    22220: 'Susono',     22221: 'Kosai',      22222: 'Izu',        22223: 'Omaezaki',
    22224: 'Kikugawa',   22225: 'Izunokuni',  22226: 'Makinohara', 22301: 'Higashiizu',
    22302: 'Kawazu',     22304: 'Minamiizu',  22305: 'Matsuzaki',  22306: 'Nishiizu',
    22325: 'Kannami',    22341: 'Shimizu',    22342: 'Nagaizumi',  22344: 'Oyama',
    22424: 'Yoshida',    22429: 'Kawanehon',  22461: 'Mori',
}


### Data-loading helpers

In [ ]:
def load(code):
    """One-row summary per configuration. Re-runs (_nox/_nops) map back to the
    original configuration so that each configuration appears once."""
    d = pd.read_csv(os.path.join(SQ, FOLD[code], 'results', 'v9_results.csv'))
    d = d[d.status == 'Optimal'].copy()
    d['cfg'] = d['tag'].str.replace(r'_(nox|nops)$', '', regex=True)
    return d.drop_duplicates('cfg', keep='last')

D = {c: load(c) for c, *_ in CLASSES}


def pick(code, gran='pl', tier2='on', **fix):
    """The single row obtained by moving only the requested parameters away from
    the baseline case.

    Unless stated explicitly, the Tier 2 acceptance cap is restricted to "no cap".
    Without that filter, t2cap-axis configurations that match the baseline in every
    other respect would also be picked up as the baseline case.
    """
    cap = fix.pop('tier2_cap', None)
    kp = fix.pop('kappa_p', np.inf); kc = fix.pop('kappa_c', 1.0)
    q = D[code]
    q = q[(q.granularity == gran) & (q.tier2 == tier2)]
    q = q[q.tier2_cap.isna()] if cap is None else q[np.isclose(q.tier2_cap, cap)]
    q = q[np.isclose(q.kappa_p, kp) | (np.isinf(q.kappa_p) & np.isinf(kp))]
    q = q[np.isclose(q.kappa_c, kc) | (np.isinf(q.kappa_c) & np.isinf(kc))]
    for k, v in {**BASE, **fix}.items():
        if k == 'delta2' and tier2 == 'off':
            continue
        q = q[np.isclose(q[k], v)]
    if len(q) > 1 and q.obj.nunique() > 1:
        raise RuntimeError('configuration is not unique: %s' % list(q.tag))
    return q.iloc[-1] if len(q) else None


def canon_dir(code, tier2='on'):
    """Canonicalised result folder for the baseline case (unmet-demand fairness and
    minimum Tier 0 already applied)."""
    suf = '_d20p75' if tier2 == 'on' else ''
    return os.path.join(SQ, FOLD[code], 'results',
                        '%sPL_gran_t2%s_kpinf_kc1_b0p5_r5_a0p5_pm0p4_d10p5%s_canon'
                        % (PRE[code], tier2, suf))


def wsum(df, by, val):
    """Aggregate S2 and S3 only, using the CVaR weights."""
    d = df[df.scenario.isin(W)].copy()
    d['w'] = d.scenario.map(W) * d[val]
    return d.groupby(by)['w'].sum()


def endlabel(ax, x, y, text, side='right'):
    ax._pending = getattr(ax, '_pending', []) + [(x, y, text, side)]


def place_labels(ax, min_gap_pt=10.0):
    """Place the end-of-line labels so that they do not overlap, keeping their order."""
    items = getattr(ax, '_pending', [])
    if not items:
        return
    fig = ax.figure; fig.canvas.draw()
    to_pt = ax.transData.transform
    pts = sorted([(to_pt((x, y))[1], x, y, t, sd) for x, y, t, sd in items], reverse=True)
    gap = min_gap_pt * fig.dpi / 72.0
    placed = []
    for py, x, y, t, sd in pts:
        if placed and placed[-1] - py < gap:
            py = placed[-1] - gap
        placed.append(py)
        ypx = to_pt((x, y))[1]
        dy = (py - ypx) * 72.0 / fig.dpi
        ax.annotate(t, (x, y), xytext=(5 if sd == 'right' else -7, dy),
                    textcoords='offset points', va='center',
                    ha='left' if sd == 'right' else 'right',
                    fontsize=7.5, color='#2b2b28',
                    arrowprops=dict(arrowstyle='-', color='#b5b5ae', lw=0.5,
                                    shrinkA=0, shrinkB=2) if abs(dy) > 2 else None)
    ax._pending = []


def show(fig, *names):
    for ax in fig.axes:
        place_labels(ax)
    savefig(fig, *names)
    plt.show()

print('configurations loaded:', {c: len(D[c]) for c in D})


def demand_share(code):
    """Share of high-risk ingredients in total demand (CVaR-weighted)."""
    du = pd.read_csv(os.path.join(SQ, FOLD[code], 'data', 'drug_usage_df.csv'))
    ids = set(du[du.ATCcode.isin(HIGH[code])].class_id)
    dm = pd.read_csv(os.path.join(SQ, FOLD[code], 'data', 'demand.csv')).rename(
        columns={'scenario_id': 'scenario'})
    return (wsum(dm[dm.class_id.isin(ids)], 'scenario', 'demand_mg').sum()
            / wsum(dm, 'scenario', 'demand_mg').sum())


---
# Section 4  Data and experimental setup

## Table 1  Structure of the three target therapeutic groups (Section 4.3)

In [ ]:
import sys
rows = []
for c, f, *_ in CLASSES:
    sys.path.insert(0, os.path.join(SQ, f))
    for m in [k for k in list(sys.modules) if k.startswith('common')]:
        del sys.modules[m]
    from common import pl_params
    d = pl_params.build(None, kappa_p=float('inf'), kappa_c=1.0)
    n1 = sum(1 for x in d.comp_pairs if x[2] == 1)
    n2 = sum(1 for x in d.comp_pairs if x[2] == 2)
    full = sum(d.cost[k] * d.cap[k] * len(d.warehouses) for k in d.products)
    ed = sum(d.probs[s] * sum(v for (n, cc, ss), v in d.demand_ck.items() if ss == s)
             for s in d.scenarios)
    rows.append(dict(Group='%s %s' % (c, NAME[c]), Products=len(d.products),
                     Ingredients=len(d.components),
                     L4_groups=len({x[:5] for x in d.components}),
                     Tier1_pairs=n1, Tier2_pairs=n2,
                     T2_T1_ratio=round(n2 / n1, 1),
                     Products_per_ingredient=round(len(d.products) / len(d.components), 1),
                     cbar=round(d.cbar, 2),
                     Expected_demand=round(ed),
                     Full_stock_cost=round(full),
                     Budget=round(d.budget_per_wh * len(d.warehouses))))
    sys.path.remove(os.path.join(SQ, f))
tbl_classes = pd.DataFrame(rows).set_index('Group')
tbl_classes

---
# Section 5  Results

## Table 2  Baseline case for the three therapeutic groups (Section 5.2)

$(\kappa_p,\kappa_c)=(\infty,1)$, $\beta=0.5$, $\gamma=5$, $r=5$, $\delta_1=0.5$,
$\delta_2=0.75$, $p_m=0.4$, $\alpha=0.5$.

In [ ]:
def unmet_cvar(code, tier2='on'):
    """CVaR-weighted unmet demand (total, high-risk)."""
    u = pd.read_csv(os.path.join(canon_dir(code, tier2), 'unmet_demand.csv'))
    return (wsum(u, 'scenario', 'unmet').sum(),
            wsum(u[u.component.isin(HIGH[code])], 'scenario', 'unmet').sum())

rows = []
for c, *_ in CLASSES:
    r = pick(c)
    al, hi = unmet_cvar(c)
    rows.append(dict(Group='%s %s' % (c, NAME[c]),
                     Objective=round(r.obj), Stockpile_cost=round(r.storage_cost),
                     CVaR=round(r.cvar), Total_stockpile=round(r.total_stock),
                     Mean_unit_price=r.mean_unit_price,
                     Unmet_CVaR_weighted=round(al), Of_which_high_risk=round(hi),
                     High_risk_share=('%.1f%%' % (100 * hi / al))))
tbl_base = pd.DataFrame(rows).set_index('Group')
tbl_base

## Figure 1 and Table 3  Unmet demand by ingredient (Section 5.2)

Colored bars are high-risk ingredients. For antidiabetics and antiepileptics no
high-risk ingredient appears among the largest unmet quantities.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.3))
tops = {}
for ax, (c, *_rest) in zip(axes, CLASSES):
    u = pd.read_csv(os.path.join(canon_dir(c), 'unmet_demand.csv'))
    s = wsum(u, 'component', 'unmet').sort_values(ascending=False)
    tops[c] = s
    t = s.head(8)[::-1]
    ax.barh(range(len(t)), t.values / 1e3,
            color=[COL[c] if i in HIGH[c] else '#c3c2b7' for i in t.index],
            edgecolor='white', linewidth=2)
    ax.set_yticks(range(len(t))); ax.set_yticklabels(t.index, fontsize=7.5)
    ax.set_title('%s (%s)' % (NAME[c], c), loc='left')
    ax.set_xlabel('Unmet demand ($10^3$ patient-days)')
    ax.grid(axis='y', visible=False)
fig.text(0.005, 0.985, 'Colored bars: high-risk ingredients (grey: others)',
         fontsize=8, color=INK2, va='top')
fig.tight_layout(rect=(0, 0, 1, 0.94))
show(fig, 'fig_unmet_by_component.pdf')

# Table: top five ingredients
rows = []
for c, *_ in CLASSES:
    s = tops[c]; tot = s.sum()
    for comp, v in s.head(5).items():
        rows.append(dict(Group=NAME[c], Ingredient=comp + ('†' if comp in HIGH[c] else ''),
                         Unmet=round(v), Share='%.1f%%' % (100 * v / tot)))
pd.DataFrame(rows).set_index(['Group', 'Ingredient'])

## Figure 2  Substitution matrix between ingredients (Section 5.2)

The diagonal is Tier 0 (substitution between products of the same ingredient). The
further from the diagonal, the more distant the substitution is clinically.

In [ ]:
def subst_matrix(code):
    d = canon_dir(code)
    g = wsum(pd.read_csv(os.path.join(d, 'substitutions.csv')),
             ['donor', 'receiver'], 'amount_allnodes').reset_index()
    t0 = wsum(pd.read_csv(os.path.join(d, 'tier0_by_component.csv')),
              'component', 'tier0_amount')
    comps = sorted(set(g.donor) | set(g.receiver) | set(t0.index))
    M = pd.DataFrame(0.0, index=comps, columns=comps)
    for r in g.itertuples(index=False):
        M.loc[r.donor, r.receiver] = r.w
    for c in comps:
        if c in t0.index:
            M.loc[c, c] = t0[c]      # diagonal = Tier 0
    return M

CODE = 'A10B'          # switch to 'N03A' / 'B01A' for the other groups
M = subst_matrix(CODE)
A = M.values; pos = A[A > 1e-9]
fig, ax = plt.subplots(figsize=(6.0, 5.2))
im = ax.imshow(np.ma.masked_where(A < 1e-9, A), cmap=BLUES,
               norm=LogNorm(vmin=max(pos.min(), 1.0), vmax=pos.max()),
               aspect='equal', interpolation='nearest')
n = len(M)
ax.set_xticks(range(n)); ax.set_xticklabels(M.columns, rotation=90, fontsize=6)
ax.set_yticks(range(n)); ax.set_yticklabels(M.index, fontsize=6)
ax.set_xticks(np.arange(-0.5, n, 1), minor=True)
ax.set_yticks(np.arange(-0.5, n, 1), minor=True)
ax.grid(which='minor', color='#ededed', linewidth=0.4); ax.grid(which='major', visible=False)
ax.tick_params(which='minor', length=0)
ax.set_xlabel('Receiving side (demand)'); ax.set_ylabel('Donating side (supply)')
ax.set_title('%s: diagonal = Tier 0 (within-ingredient product substitution)' % NAME[CODE],
             loc='left', fontsize=8.5, color=INK2)
cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cb.set_label('Substitution (patient-days, log scale)', fontsize=8)
savefig(fig, 'fig_subst_matrix_pl.pdf', 'fig_subst_matrix_pl.png')
plt.show()

## Figure 3  Unmet demand by node (Section 5.2)

Unmet demand spans two orders of magnitude across nodes, in proportion to the
population each node serves. The unmet *ratio*, by contrast, is almost constant
between nodes because of the fairness step in the canonicalisation (range 0.003-0.009).
A histogram of a near-constant quantity collapses into a single vertical line, so the
ratio is shown as a median with its range instead.

In [ ]:
def node_stats(code):
    u = pd.read_csv(os.path.join(canon_dir(code), 'unmet_demand.csv'))
    dm = pd.read_csv(os.path.join(SQ, FOLD[code], 'data', 'demand.csv')).rename(
        columns={'node_id': 'node', 'scenario_id': 'scenario'})
    t = pd.DataFrame({'unmet': wsum(u, 'node', 'unmet')}).join(
        pd.DataFrame({'demand': wsum(dm, 'node', 'demand_mg')}), how='inner')
    t['rate'] = t.unmet / t.demand
    return t

NS = {c: node_stats(c) for c, *_ in CLASSES}
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.0), gridspec_kw={'width_ratios': [1.5, 1]})
for c, *_ in CLASSES:
    t = NS[c]
    axes[0].hist(t.unmet, bins=np.logspace(np.log10(max(t.unmet.min(), 1)),
                                           np.log10(t.unmet.max()), 40),
                 histtype='step', lw=1.6, color=COL[c], label=NAME[c])
axes[0].set_xscale('log'); axes[0].margins(y=0.12)
axes[0].set_xlabel('Unmet demand per node (patient-days, log scale)'); axes[0].set_ylabel('Number of nodes')
axes[0].legend(frameon=False)
y = np.arange(len(CLASSES))[::-1]
for yi, (c, *_) in zip(y, CLASSES):
    t = NS[c]; lo, md_, hi = t.rate.min(), t.rate.median(), t.rate.max()
    axes[1].plot([lo, hi], [yi, yi], color=COL[c], lw=3, solid_capstyle='round', zorder=2)
    axes[1].plot([md_], [yi], 'o', ms=8, color='white', mec=COL[c], mew=2, zorder=3)
    axes[1].annotate('%.3f' % md_, (md_, yi), xytext=(0, 9), textcoords='offset points',
                     ha='center', fontsize=7.5)
    axes[1].annotate('range %.4f' % (hi - lo), (md_, yi), xytext=(0, -14),
                     textcoords='offset points', ha='center', fontsize=7, color=INK2)
axes[1].set_yticks(y); axes[1].set_yticklabels([NAME[c] for c, *_ in CLASSES])
axes[1].set_xlabel('Unmet ratio per node (min-max / median)')
axes[1].grid(axis='y', visible=False)
axes[1].set_ylim(-0.6, len(CLASSES) - 0.3)
axes[1].set_xlim(0, max(NS[c].rate.max() for c, *_ in CLASSES) * 1.28)
fig.tight_layout(); savefig(fig, 'fig_node_unmet.pdf'); plt.show()

## Figure 4  Spatial distribution of unmet demand (Section 5.2)

The unmet ratio is almost constant across nodes and is therefore not colour-encoded:
stretching a range of 0.007 over the full colour scale would suggest regional
differences that do not exist. What the map can show is the unmet *quantity*, that is,
the spatial pattern of demand density.

In [ ]:
CODE = 'A10B'
t = NS[CODE].join(pd.read_csv(os.path.join(SQ, FOLD[CODE], 'data', 'nodes.csv'))
                    .set_index('node_id')[['latitude', 'longitude']], how='inner')
fig, ax = plt.subplots(figsize=(8.0, 4.4))
sc = ax.scatter(t.longitude, t.latitude, s=14, c=t.unmet, cmap=BLUES,
                norm=LogNorm(vmin=max(t.unmet.min(), 1), vmax=t.unmet.max()),
                linewidths=0.25, edgecolors='#8a8a85', alpha=0.9)
cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
cb.set_label('Unmet demand per node (patient-days, log scale)', fontsize=8)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_aspect(1 / np.cos(np.deg2rad(t.latitude.mean())))
ax.set_title('%s: the unmet ratio is almost constant across nodes, so it is not colour-encoded' % NAME[CODE],
             loc='left', fontsize=8.5, color=INK2)
plt.show()

## Figure 5  Sensitivity analysis: budget ratio $\beta$ and substitution allowance ratio $p_m$ (Sections 5.4 and 5.5)

In [ ]:
def sens_fig(key, values, base_v, xlabel, xticks=None, fname=None):
    fig, ax = plt.subplots(figsize=(5.6, 3.2))
    ax.axhline(0, color='#b5b5ae', lw=0.8, zorder=1)
    for c, *_ in CLASSES:
        xs, ys = [], []
        for v in values:
            r = pick(c, **{key: v})
            if r is not None:
                xs.append(v); ys.append(r.cvar)
        b = pick(c, **{key: base_v}).cvar
        rel = 100 * (np.array(ys) / b - 1)
        ax.plot(xs, rel, color=COL[c], lw=1.6, marker=MK[c], ms=6,
                mec='white', mew=1.0, zorder=3)
        endlabel(ax, xs[-1], rel[-1], NAME[c])
    ax.set_xlabel(xlabel); ax.set_ylabel('Change in CVaR (%, vs. baseline)')
    if xticks is not None:
        ax.set_xticks(xticks)
    lo, hi = min(values), max(values)
    ax.set_xlim(lo - 0.04 * (hi - lo), hi + 0.62 * (hi - lo))
    show(fig, *( (fname,) if fname else () ))

sens_fig('budget_rate', [0.05, 0.15, 0.3, 0.5, 0.7, 1.0], 0.5,
         r'Budget ratio $\beta$', [0.05, 0.3, 0.5, 0.7, 1.0], 'fig_sens_budget.pdf')
sens_fig('p_m', [0.2, 0.4, 0.6, 0.8], 0.4,
         r'Substitution allowance ratio $p_m$', [0.2, 0.4, 0.6, 0.8], 'fig_sens_pm.pdf')

## Figure 6  High-risk weight $r$ (Section 5.8)

The vertical axis is the realised allocation, not the CVaR. Because $r$ rescales the
unmet-demand weight $\omega$ itself, the CVaR moves in proportion to $r$ even when the
allocation is unchanged, so a change in CVaR cannot be read as stronger protection.

Also note that `E_unmet_high` in `v9_results.csv` identifies high-risk ingredients as
those whose $\omega$ exceeds the base value. At $r=1$ every ingredient has the same
$\omega$, the set is empty and the column is recorded as 0. Here the high-risk set is
taken from a fixed list instead.

In [ ]:
R = [1.0, 3.0, 5.0, 7.5, 10.0]
fig, ax = plt.subplots(figsize=(5.6, 3.1))
for c, *_ in CLASSES:
    ys = []
    for r in R:
        tag = ('%sPL_gran_t2on_kpinf_kc1_b0p5_r5_a0p5_pm0p4_d10p5_d20p75' % PRE[c]
               if r == 5.0 else
               '%sPL_rhigh_t2on_kpinf_kc1_b0p5_r%s_a0p5_pm0p4_d10p5_d20p75'
               % (PRE[c], ('%g' % r).replace('.', 'p')))
        u = pd.read_csv(os.path.join(SQ, FOLD[c], 'results', tag, 'unmet_demand.csv'))
        ys.append(wsum(u[u.component.isin(HIGH[c])], 'scenario', 'unmet').sum())
    rel = 100 * np.array(ys) / ys[0]
    ax.plot(R, rel, color=COL[c], lw=1.6, marker=MK[c], ms=6, mec='white', mew=1.0, zorder=3)
    endlabel(ax, R[-1], rel[-1], '%s %.0f%%' % (NAME[c], rel[-1]))
ax.set_xticks(R); ax.set_xlabel(r'High-risk weight multiplier $r$')
ax.set_ylabel('High-risk unmet demand (%, vs. $r=1$)')
ax.set_ylim(0, 108); ax.set_xlim(0.4, 19.5)
show(fig, 'fig_sens_rhigh.pdf')
print('For r >= 3 the allocation no longer changes in any of the three groups:')
print('the high-risk ingredients are already stocked up to the capacity limit, so protection saturates.')

## Figure 7  Penalty coefficient $\delta_2$ (Section 5.6)

The Tier 2 substitution volume in $S2$ and $S3$ does not move by a single patient-day in
any of the three groups: the price elasticity is exactly zero, and all that changes is
the price paid. Aggregated with the scenario probabilities (including $S1$), the volume
would appear to *increase* as $\delta_2$ rises, which is contradictory; the cell below
shows both aggregations side by side.

In [ ]:
D2 = [0.75, 1.0, 1.5, 2.0]
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.0))
chk = []
for c, *_ in CLASSES:
    q23, cv, qall = [], [], []
    for d2 in D2:
        tag = ('%sPL_gran_t2on_kpinf_kc1_b0p5_r5_a0p5_pm0p4_d10p5_d20p75' % PRE[c]
               if d2 == 0.75 else
               '%sPL_delta_t2on_kpinf_kc1_b0p5_r5_a0p5_pm0p4_d10p5_d2%s'
               % (PRE[c], ('%g' % d2).replace('.', 'p')))
        sb = pd.read_csv(os.path.join(SQ, FOLD[c], 'results', tag, 'substitutions.csv'))
        t2 = sb[sb.tier == 2]
        q23.append(wsum(t2, 'scenario', 'amount_allnodes').sum())
        p = {'S1': 0.5, 'S2': 0.35, 'S3': 0.15}
        g = t2.groupby('scenario').amount_allnodes.sum()
        qall.append(sum(p[k] * g.get(k, 0) for k in p))
        cv.append(pick(c, delta2=d2).cvar)
    q23, cv, qall = map(np.array, (q23, cv, qall))
    axes[0].plot(D2, 100 * (q23 / q23[0] - 1), color=COL[c], lw=1.6, marker=MK[c],
                 ms=6, mec='white', mew=1.0, zorder=3)
    axes[1].plot(D2, 100 * (cv / cv[0] - 1), color=COL[c], lw=1.6, marker=MK[c],
                 ms=6, mec='white', mew=1.0, zorder=3)
    endlabel(axes[1], D2[-1], 100 * (cv[-1] / cv[0] - 1), NAME[c])
    chk.append(dict(Group=NAME[c],
                    S2S3=('%+.1f%%' % (100 * (q23[-1] / q23[0] - 1))),
                     Prob_weighted_incl_S1=('%+.1f%%' % (100 * (qall[-1] / qall[0] - 1))),
                    CVaR=('%+.1f%%' % (100 * (cv[-1] / cv[0] - 1)))))
axes[0].set_ylim(-10, 10); axes[0].set_title('Tier 2 substitution (S2 and S3)', loc='left')
axes[0].set_ylabel(r'Change (%, vs. $\delta_2=0.75$)')
axes[0].annotate('0.0% in all three groups', (1.375, 0), xytext=(0, 8), textcoords='offset points',
                 ha='center', fontsize=7.5, color=INK2)
axes[1].set_title('CVaR', loc='left'); axes[1].set_xlim(0.65, 2.55)
for ax in axes:
    ax.set_xticks(D2); ax.set_xlabel(r'Tier 2 penalty coefficient $\delta_2$ ($\delta_1=0.5$)')
fig.tight_layout(); show(fig, 'fig_delta.pdf')
print('Change in Tier 2 substitution when delta2 goes from 0.75 to 2.0 (x2.67)')
display(pd.DataFrame(chk).set_index('Group'))

## Figure 8  CVaR confidence level $\alpha$ and the value of Tier 2 (Section 5.7)

The higher $\alpha$, the smaller the loss from banning Tier 2; for antithrombotics it
reaches exactly zero at $\alpha=0.9$. In the worst case every ingredient is short at the
same time, so there is no slack left to move between classes. The ranking of the three
groups is nevertheless preserved at every $\alpha$.

In [ ]:
A = [0.3, 0.5, 0.7, 0.9]
fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.axhline(0, color='#b5b5ae', lw=0.8, zorder=1)
for c, *_ in CLASSES:
    ys = [100 * (pick(c, 'pl', 'off', alpha=a).obj / pick(c, 'pl', 'on', alpha=a).obj - 1)
          for a in A]
    ax.plot(A, ys, color=COL[c], lw=1.6, marker=MK[c], ms=6, mec='white', mew=1.0, zorder=3)
    endlabel(ax, A[-1], ys[-1], '%s (%.1f%%)' % (NAME[c], ys[-1]))
ax.set_xticks(A); ax.set_xlabel(r'CVaR confidence level $\alpha$')
ax.set_ylabel('Objective loss under Tier 2 ban (%)'); ax.set_xlim(0.26, 1.42)
show(fig, 'fig_tier2_alpha.pdf')

# The stockpile plan is invariant in alpha: at alpha=0.9 the CVaR equals the S3
# scenario cost of the baseline case.
rows = [dict(Group=NAME[c], CVaR_alpha09=round(pick(c, alpha=0.9).cvar),
             Baseline_S3_cost=round(pick(c).S3_cost),
             Diff=round(pick(c, alpha=0.9).cvar - pick(c).S3_cost))
        for c, *_ in CLASSES]
pd.DataFrame(rows).set_index('Group')

## Figure 9  Isolating the product-choice effect by moving $\kappa_p$ (Section 5.9)

The class-level cap $\kappa_c=1$ keeps the total stock of each ingredient fixed, so every
improvement obtained by relaxing $\kappa_p$ comes from the freedom of product choice
within an ingredient.


In [ ]:
KP = [1.0, 2.0, 5.0, np.inf]
X = np.arange(len(KP))
fig, ax = plt.subplots(figsize=(5.6, 3.3))
ax.axhline(0, color='#b5b5ae', lw=0.8, zorder=1)
# Only the PL curves are drawn. In the ATC model a drug class coincides with an
# ingredient, so the class-level cap reduces to the product-level cap and the curve
# is flat at 0.0% for all three groups; plotting it adds nothing but a dashed line
# lying on the zero axis (see the table below for the figures).
for c, *_ in CLASSES:
    ys = np.array([pick(c, 'pl', 'on', kappa_p=k, kappa_c=1.0).obj for k in KP])
    rel = 100 * (ys / ys[0] - 1)
    ax.plot(X, rel, color=COL[c], lw=1.6, marker=MK[c], ms=6,
            mec='white', mew=1.0, zorder=3)
    endlabel(ax, X[-1], rel[-1], '%s' % NAME[c])
ax.set_xticks(X); ax.set_xticklabels(['1', '2', '5', '∞'])
ax.set_xlabel(r'Product-level procurement multiplier $\kappa_p$ ($\kappa_c=1$ fixed)')
ax.set_ylabel(r'Change in the objective (%, vs. $\kappa_p=1$)')
# ax.set_xlim(-0.25, len(KP) - 0.25 + 1.1)
# ax.legend(handles=[Line2D([], [], color=INK2, lw=1.6, ls='-', label='PL model'),
#                    Line2D([], [], color=INK2, lw=1.6, ls=(0, (4, 2)), marker='o',
#                           mfc='white', mec=INK2, label='ATC model')],
#           loc='upper right', bbox_to_anchor=(1.0, 0.86), frameon=False)
show(fig, 'fig_kappa_p.pdf', 'fig_kappa_p.png')

# Table: kappa vs. the objective, and the decomposition of the effect
rows = []
for c, *_ in CLASSES:
    p1 = pick(c, 'pl', 'on', kappa_p=1.0, kappa_c=1.0).obj
    pi = pick(c, 'pl', 'on', kappa_p=np.inf, kappa_c=1.0).obj
    pii = pick(c, 'pl', 'on', kappa_p=np.inf, kappa_c=np.inf).obj
    a1 = pick(c, 'atc', 'on', kappa_p=1.0, kappa_c=1.0).obj
    ai = pick(c, 'atc', 'on', kappa_p=np.inf, kappa_c=1.0).obj
    aii = pick(c, 'atc', 'on', kappa_p=np.inf, kappa_c=np.inf).obj
    rows.append(dict(Group=NAME[c],
                     PL_granularity=('%+.2f%%' % (100 * (pi / p1 - 1))),
                     ATC_granularity=('%+.2f%%' % (100 * (ai / a1 - 1))),
                     PL_cross_class=('%+.1f%%' % (100 * (pii / pi - 1))),
                     ATC_cross_class=('%+.1f%%' % (100 * (aii / ai - 1)))))
print('Decomposition of the improvement from relaxing kappa (change in the objective)')
display(pd.DataFrame(rows).set_index('Group'))

## Figure 10  Share of the $\kappa_p$ effect recovered (Section 5.9)

At $\kappa_p=2$, that is, with twice the routine consumption available, 56-87% of the
attainable improvement is already obtained. Unlimited procurement is not needed to
realise the value of product-level granularity.

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.1))
for c, *_ in CLASSES:
    ys = np.array([pick(c, 'pl', 'on', kappa_p=k, kappa_c=1.0).obj for k in KP])
    cap = 100 * (ys[0] - ys) / (ys[0] - ys[-1])
    ax.plot(X, cap, color=COL[c], lw=1.6, marker=MK[c], ms=6, mec='white', mew=1.0, zorder=3)
    # All three series converge to 100% at the right edge, so label them at
    # kappa_p = 2, where they are furthest apart.
    endlabel(ax, X[1], cap[1], '%s %.0f%%' % (NAME[c], cap[1]), side='left')
ax.set_xticks(X); ax.set_xticklabels(['1', '2', '5', '∞'])
ax.set_xlabel(r'Product-level procurement multiplier $\kappa_p$')
ax.set_ylabel('Improvement recovered (%)')
# ax.set_title(r'Share of the $\kappa_p\to\infty$ improvement recovered at each $\kappa_p$',
#              loc='left', fontsize=8.5, color=INK2)
ax.set_ylim(-5, 108); ax.set_xlim(-0.25, len(KP) - 0.10)
show(fig, 'fig_kappa_capture.pdf', 'fig_kappa_capture.png')

## Table 6 and Figure 11  The value of Tier 2 differs greatly between groups (Sections 5.11 and 6.2)

The deterioration under a ban ranges from 26.0% for antidiabetics to 1.9% for
antithrombotics, a spread of more than a factor of 13. The ranking follows the ratio of
the number of Tier 2 pairs to the number of Tier 1 pairs.

Banning Tier 2 also leaves the **total** unmet demand unchanged. Total supply is fixed by
the budget; substitution does not increase it but redirects it towards the high-risk
ingredients.

The mix under the ban is shown alongside. **Tier 1 is identical with and without the
ban** (to the decimal). Since $\pi_1=2.5\bar c < \pi_2=3.75\bar c <$ unmet demand at
$5\bar c$, the model exhausts the cheapest option, Tier 1, first and lets Tier 2 absorb
what is left. Tier 1 is therefore already at its cap and cannot grow when Tier 2 is
banned; what absorbs the lost Tier 2 is the redistribution of Tier 0 and unmet demand.

In [ ]:
def subst_mix(code, tier2):
    """CVaR-weighted substitution volumes (Tier 0 / Tier 1 / Tier 2)."""
    d = canon_dir(code, tier2)
    t0 = wsum(pd.read_csv(os.path.join(d, 'tier0_by_component.csv')),
              'scenario', 'tier0_amount').sum()
    sb = pd.read_csv(os.path.join(d, 'substitutions.csv'))
    t1 = wsum(sb[sb.tier == 1], 'scenario', 'amount_allnodes').sum()
    t2 = wsum(sb[sb.tier == 2], 'scenario', 'amount_allnodes').sum()
    return t0, t1, t2

rows, mix = [], []
for c, *_ in CLASSES:
    on, off = pick(c, 'pl', 'on'), pick(c, 'pl', 'off')
    ao, ho = unmet_cvar(c, 'on'); af, hf = unmet_cvar(c, 'off')
    rows.append(dict(Group=NAME[c],
                     Objective=('%+.1f%%' % (100 * (off.obj / on.obj - 1))),
                     CVaR=('%+.1f%%' % (100 * (off.cvar / on.cvar - 1))),
                     Unmet_total=('%+.1f%%' % (100 * (af / ao - 1))),
                     High_risk_unmet=('%+.1f%%' % (100 * (hf / ho - 1)))))
    for tier2, lab in [('on', 'allowed'), ('off', 'banned')]:
        t0, t1, t2 = subst_mix(c, tier2)
        tot = t0 + t1 + t2
        mix.append(dict(Group=NAME[c], Tier2=lab,
                        Tier0=round(t0), Tier1=round(t1), Tier2_amount=round(t2),
                        Tier0_share=('%.1f%%' % (100 * t0 / tot)),
                        Tier1_share=('%.1f%%' % (100 * t1 / tot)),
                        Tier2_share=('%.1f%%' % (100 * t2 / tot))))
print('Impact of banning Tier 2')
display(pd.DataFrame(rows).set_index('Group'))
print()
print('Substitution mix (Tier 2 allowed / banned)')
display(pd.DataFrame(mix).set_index(['Group', 'Tier2']))
print()
print('Is Tier 1 identical with and without the ban (to the decimal)?')
for c, *_ in CLASSES:
    a = subst_mix(c, 'on')[1]; b = subst_mix(c, 'off')[1]
    print('  %-16s allowed %.6f  banned %.6f  diff %.2e' % (NAME[c], a, b, abs(a - b)))

## Figure 12  Checking the prediction (Section 6.2)

Relaxing $\kappa_c$ works only because Tier 2 substitution is permitted; it is not an
independent lever. That yields the prediction that the smaller the value of Tier 2 in a
therapeutic group, the smaller the benefit of $\kappa_c$. If the ranking of the groups
agrees between the two panels, the prediction holds.

In [ ]:
t2, kc = [], []
for c, *_ in CLASSES:
    on, off = pick(c, 'pl', 'on'), pick(c, 'pl', 'off')
    k = pick(c, 'pl', 'on', kappa_p=np.inf, kappa_c=np.inf)
    t2.append(100 * (off.obj / on.obj - 1))
    kc.append(100 * (1 - k.obj / on.obj))
fig, axes = plt.subplots(1, 2, figsize=(9.0, 2.8))
xmax = max(max(t2), max(kc)) * 1.22
for ax, vals, title in [(axes[0], t2, 'Deterioration under a Tier 2 ban'),
                        (axes[1], kc, r'Improvement from relaxing $\kappa_c$')]:
    y = np.arange(len(CLASSES))[::-1]
    for yi, v, (c, *_) in zip(y, vals, CLASSES):
        ax.barh(yi, v, height=0.55, color=COL[c], edgecolor='white', linewidth=2)
        ax.text(v + xmax * 0.02, yi, '%.1f%%' % v, va='center', fontsize=8)
    ax.set_yticks(y); ax.set_yticklabels([NAME[c] for c, *_ in CLASSES])
    ax.set_title(title, loc='left'); ax.set_xlim(0, xmax)
    ax.grid(axis='y', visible=False); ax.set_xlabel('%')
fig.tight_layout(); savefig(fig, 'fig_prediction.pdf'); plt.show()

order_t2 = [NAME[c] for _, c in sorted(zip(t2, [c for c, *_ in CLASSES]), reverse=True)]
order_kc = [NAME[c] for _, c in sorted(zip(kc, [c for c, *_ in CLASSES]), reverse=True)]
print('Ranking by the value of Tier 2 :', ' > '.join(order_t2))
print('Ranking by the benefit of κ_c  :', ' > '.join(order_kc))
print('Rankings agree                 :', order_t2 == order_kc)

---
# Additional figures

## Figure 13  Unmet demand by municipality, with boundaries

The boundary data are the National Land Numerical Information administrative area data
(N03, 2021-01-01), simplified by 1%. Shizuoka City and Hamamatsu City are stored by
ward, so they are dissolved to the city-level codes used on the node side
(22100 / 22130).

Colour encodes the unmet quantity. The unmet ratio is almost constant between
municipalities as well, so it is not colour-encoded.

In [ ]:
import geopandas as gpd

GEO = os.path.join(os.path.dirname(SQ), 'data', '境界', 'shizuoka_municipality.geojson')

def municipalities():
    """Return the municipality polygons, dissolved to the city-level codes used on
    the node side. Shizuoka and Hamamatsu are stored by ward in the source file."""
    g = gpd.read_file(GEO)
    g['code'] = g.N03_007.astype(int)
    g['muni'] = g.code.apply(lambda c: 22100 if 22101 <= c <= 22110
                             else (22130 if 22131 <= c <= 22140 else c))
    g['name'] = g.apply(lambda r: '静岡市' if r.muni == 22100
                        else ('浜松市' if r.muni == 22130 else (r.N03_004 or r.N03_003)), axis=1)
    out = g.dissolve(by=['muni', 'name']).reset_index()[['muni', 'name', 'geometry']]
    # Romanised labels for the figures; fall back to the source name if missing.
    out['name_en'] = out.muni.map(MUNI_EN).fillna(out['name'])
    return out


def by_municipality(code):
    """Aggregate the per-node unmet demand to municipalities."""
    nd = pd.read_csv(os.path.join(SQ, FOLD[code], 'data', 'nodes.csv')).set_index('node_id')
    t = NS[code].join(nd[['municipality_id']], how='inner')
    a = t.groupby('municipality_id')[['unmet', 'demand']].sum()
    a['rate'] = a.unmet / a.demand
    a['nodes'] = t.groupby('municipality_id').size()
    return a

CODE = 'A10B'
MUNI = municipalities()
agg = by_municipality(CODE)
gm = MUNI.merge(agg, left_on='muni', right_index=True, how='left')

fig, ax = plt.subplots(figsize=(9.0, 5.2))
gm.plot(column='unmet', cmap=BLUES, norm=LogNorm(vmin=gm.unmet.min(), vmax=gm.unmet.max()),
        edgecolor='#5a5a55', linewidth=0.6, ax=ax,
        legend=True, legend_kwds={'label': 'Unmet demand by municipality (patient-days, log scale)',
                                  'shrink': 0.7})
# Label the six municipalities with the largest unmet demand
for _, r in gm.nlargest(6, 'unmet').iterrows():
    c = r.geometry.representative_point()
    ax.annotate(r['name_en'], (c.x, c.y), ha='center', fontsize=7.5, color='#2b2b28',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.75))
ax.set_axis_off()
ax.set_title('%s: unmet demand by municipality (baseline case, CVaR-weighted)' % NAME[CODE],
             loc='left', fontsize=9.5)
savefig(fig, 'fig_node_unmet_map.png')
plt.show()

## Figure 14  Bar chart by municipality

Unmet demand differs by more than two orders of magnitude between municipalities,
directly reflecting differences in population. The unmet ratio, by contrast, is almost
constant across municipalities, showing that the fairness step in the canonicalisation
holds across the whole prefecture.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 6.2), sharey=True,
                         gridspec_kw={'width_ratios': [1.6, 1]})
a = agg.join(MUNI.set_index('muni')[['name', 'name_en']]).sort_values('unmet')
y = np.arange(len(a))
axes[0].barh(y, a.unmet / 1e3, color=COL[CODE], edgecolor='white', linewidth=1.2)
axes[0].set_yticks(y); axes[0].set_yticklabels(a['name_en'], fontsize=7.5)
axes[0].set_xlabel('Unmet demand ($10^3$ patient-days)')
axes[0].set_title('Unmet demand (over two orders of magnitude between municipalities)', loc='left')
axes[0].grid(axis='y', visible=False)
axes[1].barh(y, a.rate, color='#c3c2b7', edgecolor='white', linewidth=1.2)
axes[1].set_xlabel('Unmet ratio')
axes[1].set_title('Unmet ratio (almost constant)', loc='left')
axes[1].grid(axis='y', visible=False)
axes[1].set_xlim(0, a.rate.max() * 1.18)
axes[1].annotate('%.3f to %.3f (range %.4f)' % (a.rate.min(), a.rate.max(),
                                            a.rate.max() - a.rate.min()),
                 (a.rate.max(), len(a) - 1), xytext=(6, 0), textcoords='offset points',
                 va='center', fontsize=8, color=INK2)
fig.suptitle('%s: unmet demand by municipality (baseline case, CVaR-weighted)' % NAME[CODE], x=0.01, ha='left')
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

display(a[['nodes', 'unmet', 'demand', 'rate']].sort_values('unmet', ascending=False)
         .head(10).round({'unmet': 0, 'demand': 0, 'rate': 4}))

## Figure 15  Tier 0 substitution matrix (product by product)

This view shows, at product granularity, the reallocation happening inside an
ingredient pool. The number of products per ingredient varies a lot, so the ingredients
with the largest Tier 0 volumes are drawn in separate panels.

Note that the ingredient-level total is uniquely determined, but **the breakdown between
products is not** (a degenerate transportation problem). Here the breakdown apportions
the flow from products in surplus to products in deficit in equal proportion: the total
is exact, the breakdown is a canonical solution.

Aggregation uses the same **CVaR weights** as every other figure (0.7 on S2, 0.3 on S3).
S1 lies outside the objective and its second-stage allocation is not uniquely
determined, so mixing in the near-normal S1 would put quantities the optimiser never
decided into the figure (Section 5.4). `tier0_by_product.csv` is stored per scenario and
unweighted, so the weights are applied here.

In [ ]:
def tier0_products(code, tier2='on'):
    """Product-by-product breakdown of Tier 0 substitution (CVaR-weighted).

    tier0_by_product.csv is stored per scenario and unweighted. The second-stage
    allocation of S1 is not uniquely determined by the objective (Section 5.4),
    so, as in every other figure, only S2 and S3 are aggregated with the CVaR
    weights.
    """
    f = os.path.join(canon_dir(code, tier2), 'tier0_by_product.csv')
    if not os.path.exists(f):
        raise SystemExit('tier0_by_product.csv not found: %s' % f)
    t = pd.read_csv(f)
    if 'scenario' not in t.columns:
        raise SystemExit(
            'tier0_by_product.csv is in the old format (probability-weighted and\n'
            'already aggregated over scenarios). Re-run postprocess_canonical.py: %s' % f)
    t = wsum(t, ['component', 'donor_product', 'receiver_product'], 'amount')
    return t[t > 1e-9].rename('amount').reset_index()

CODE = 'A10B'
TOP_N = 6                     # how many ingredients (largest Tier 0) to draw
tp = tier0_products(CODE)
tot = tp.groupby('component').amount.sum().sort_values(ascending=False)
comps = list(tot.index[:TOP_N])
print('ingredients shown:', ', '.join('%s (%.0f)' % (c, tot[c]) for c in comps))

mats = []
for c in comps:
    d = tp[tp.component == c]
    prods = sorted(set(d.donor_product) | set(d.receiver_product))
    M = pd.DataFrame(0.0, index=prods, columns=prods)
    for r in d.itertuples(index=False):
        M.loc[r.donor_product, r.receiver_product] = r.amount
    mats.append((c, M))

allv = np.concatenate([m.values.ravel() for _, m in mats])
pos = allv[allv > 1e-9]
vmin, vmax = max(pos.min(), 1.0), allv.max()
ncol = min(3, len(mats)); nrow = int(np.ceil(len(mats) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 4.4 * nrow))
axes = np.atleast_1d(axes).ravel()
im = None
for ax, (c, M) in zip(axes, mats):
    A = M.values
    im = ax.imshow(np.ma.masked_where(A < vmin, A), cmap=BLUES,
                   norm=LogNorm(vmin=vmin, vmax=vmax), aspect='equal',
                   interpolation='nearest')
    n = len(M)
    if n <= 30:
        ax.set_xticks(range(n)); ax.set_xticklabels(M.columns, rotation=90, fontsize=5)
        ax.set_yticks(range(n)); ax.set_yticklabels(M.index, fontsize=5)
    else:
        ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel('%d products' % n, fontsize=7)
    ax.set_xticks(np.arange(-0.5, n, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n, 1), minor=True)
    ax.grid(which='minor', color='#ededed', linewidth=0.3); ax.grid(which='major', visible=False)
    ax.tick_params(which='minor', length=0)
    ax.set_title('%s (%d products, total %.0f)' % (c, n, M.values.sum()), fontsize=8.5)
for ax in axes[len(mats):]:
    ax.axis('off')
fig.suptitle('%s: Tier 0 substitution (product to product within an ingredient, CVaR-weighted)' % NAME[CODE],
             x=0.01, ha='left')
cb = fig.colorbar(im, ax=axes.tolist(), shrink=0.65, pad=0.02)
cb.set_label('Substitution (patient-days, log scale)', fontsize=8)
plt.show()

## Figure 16  Tier 0 substitution matrix (product names)

The same content as Figure 15, labelled with product names instead of the internal IDs
(C\_xxx). The mapping table is `product_map.csv`, produced by
`tools/make_product_map.py`; it also carries the NHI drug price code, the strength, the
manufacturer and the brand/generic flag.

Labels are prefixed with **[B]** for brand-name and **[G]** for generic products, and the
drug price is appended, so that it is possible to read off which generic products act as
the sources. Product names are kept in the original Japanese of the drug price listing.

In [ ]:
PMAP = pd.read_csv(os.path.join(SQ, 'product_map.csv'))

def label_map(code, with_price=True):
    """class_id -> figure label ([B]rand / [G]eneric, product name, drug price).

    The product names are kept as they appear in the Japanese drug price listing,
    because they have no official English equivalent.
    """
    m = PMAP[PMAP.薬効群.str.startswith(code)]
    out = {}
    for r in m.itertuples(index=False):
        tag = '[B]' if r.後発品 == '先発' else ('[G]' if r.後発品 == '後発' else '[?]')
        s = '%s %s' % (tag, r.短縮名)
        if with_price:
            s += '  %.1f JPY' % r.薬価
        out[r.class_id] = s
    return out


CODE = 'A10B'
COMPS = ['A10BA02', 'A10BH01', 'A10BH08']   # ingredients with a manageable number of products
LAB = label_map(CODE)
tp = tier0_products(CODE)

mats = []
for c in COMPS:
    d = tp[tp.component == c]
    prods = sorted(set(d.donor_product) | set(d.receiver_product))
    M = pd.DataFrame(0.0, index=prods, columns=prods)
    for r in d.itertuples(index=False):
        M.loc[r.donor_product, r.receiver_product] = r.amount
    # Sorting by outflow makes it easy to read off which products are the sources
    order = M.sum(axis=1).sort_values(ascending=False).index
    mats.append((c, M.loc[order, order]))

allv = np.concatenate([m.values.ravel() for _, m in mats])
pos = allv[allv > 1e-9]
vmin, vmax = max(pos.min(), 1.0), allv.max()

for c, M in mats:
    n = len(M)
    fig, ax = plt.subplots(figsize=(max(5.5, 0.34 * n + 4.2), max(4.2, 0.30 * n + 2.6)))
    A = M.values
    im = ax.imshow(np.ma.masked_where(A < vmin, A), cmap=BLUES,
                   norm=LogNorm(vmin=vmin, vmax=vmax), aspect='equal',
                   interpolation='nearest')
    lab = [LAB.get(p, p) for p in M.index]
    ax.set_xticks(range(n)); ax.set_xticklabels(lab, rotation=90, fontsize=6.5)
    ax.set_yticks(range(n)); ax.set_yticklabels(lab, fontsize=6.5)
    ax.set_xticks(np.arange(-0.5, n, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n, 1), minor=True)
    ax.grid(which='minor', color='#ededed', linewidth=0.3); ax.grid(which='major', visible=False)
    ax.tick_params(which='minor', length=0)
    ax.set_xlabel('Receiving side'); ax.set_ylabel('Donating side (ordered by outflow)')
    comp_name = PMAP[(PMAP.薬効群.str.startswith(CODE)) & (PMAP.ATCcode == c)].成分名.iloc[0]
    ax.set_title('%s  %s (%d products, total %.0f patient-days, CVaR-weighted)'
                 % (c, comp_name, n, M.values.sum()), loc='left', fontsize=9)
    cb = fig.colorbar(im, ax=ax, shrink=0.75, pad=0.02)
    cb.set_label('Substitution (patient-days, log scale)', fontsize=8)
    if c == 'A10BA02':
        savefig(fig, 'tier0_A10BA02.png')
    plt.show()

### Products with the largest substitution outflow

A list of which products act as the sources of Tier 0 substitution. The outflow is in
patient-days, aggregated with the same CVaR weights as Figures 15 and 16
(0.7 on S2, 0.3 on S3).

In [ ]:
rows = []
for c in COMPS:
    d = tp[tp.component == c]
    don = d.groupby('donor_product').amount.sum().sort_values(ascending=False)
    info = PMAP[PMAP.薬効群.str.startswith(CODE)].set_index('class_id')
    for cid, v in don.head(3).items():
        r = info.loc[cid]
        rows.append(dict(Ingredient=c, class_id=cid, Product=r.医薬品名,
                         NHI_code=r.薬価基準収載医薬品コード,
                         Price_JPY=r.薬価, Brand_or_generic=r.後発品, Outflow=round(v)))
tbl_supply = pd.DataFrame(rows).set_index(['Ingredient', 'class_id'])
print('Substitution outflow in patient-days (CVaR-weighted: S2 0.7 / S3 0.3)')
for c in COMPS:
    print('  %s within-ingredient Tier 0 total = %.0f' % (c, tp[tp.component == c].amount.sum()))
display(tbl_supply)

## Figure 17  The price of protecting high-risk ingredients ($r=1 \to 3$)

Raising $r$ leaves the **total** unmet demand unchanged: what is taken off the high-risk
ingredients moves one-for-one to the others (for antidiabetics $-561{,}185$ and
$+561{,}185$, a difference of zero). Total supply is fixed by the budget, so all $r$ can
move is the split.

The burden carried by the other ingredients decomposes exactly as

$$\text{increase in other ingredients} \;=\; \frac{h_1}{o_1} \times \left(1 - \frac{h_3}{h_1}\right)$$

where $h$ and $o$ are the unmet demand of the high-risk and the other ingredients and the
subscript is the value of $r$. It is thus the product of "how much larger the high-risk
unmet demand is than that of the other ingredients at $r=1$" and "how much of it can be
removed".

In [ ]:
R_PAIR = [1.0, 3.0]

def rhigh_split(code, r):
    """Unmet demand of high-risk vs. other ingredients (CVaR-weighted)."""
    tag = ('%sPL_gran_t2on_kpinf_kc1_b0p5_r5_a0p5_pm0p4_d10p5_d20p75' % PRE[code]
           if r == 5.0 else
           '%sPL_rhigh_t2on_kpinf_kc1_b0p5_r%s_a0p5_pm0p4_d10p5_d20p75'
           % (PRE[code], ('%g' % r).replace('.', 'p')))
    u = pd.read_csv(os.path.join(SQ, FOLD[code], 'results', tag, 'unmet_demand.csv'))
    hi = wsum(u[u.component.isin(HIGH[code])], 'scenario', 'unmet').sum()
    ot = wsum(u[~u.component.isin(HIGH[code])], 'scenario', 'unmet').sum()
    return hi, ot

S = {c: {r: rhigh_split(c, r) for r in R_PAIR} for c, *_ in CLASSES}

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.4), gridspec_kw={'width_ratios': [1.35, 1]})

# Left panel: stacked bars show that the total is conserved and only the split moves
x = np.arange(len(CLASSES) * 2 + 1, dtype=float)
x = np.concatenate([[0, 1], [3, 4], [6, 7]])
lbl, hi_v, ot_v = [], [], []
for c, *_ in CLASSES:
    for r in R_PAIR:
        h, o = S[c][r]
        hi_v.append(h / 1e3); ot_v.append(o / 1e3); lbl.append('$r$=%g' % r)
ax = axes[0]
ax.bar(x, ot_v, width=0.8, color='#c3c2b7', edgecolor='white', linewidth=2, label='Other ingredients')
ax.bar(x, hi_v, width=0.8, bottom=ot_v, color='#2a78d6', edgecolor='white', linewidth=2,
       label='High-risk ingredients')
for i in range(0, len(x), 2):
    tot0 = ot_v[i] + hi_v[i]; tot1 = ot_v[i + 1] + hi_v[i + 1]
    ax.plot([x[i] - 0.4, x[i + 1] + 0.4], [tot0, tot1], color=INK2, lw=1.0, ls=(0, (3, 2)),
            zorder=4)
    ax.annotate('total %+.1f%%' % (100 * (tot1 / tot0 - 1)), ((x[i] + x[i + 1]) / 2, tot0),
                xytext=(0, 7), textcoords='offset points', ha='center', fontsize=7.5,
                color=INK2)
ax.set_xticks(x); ax.set_xticklabels(lbl, fontsize=8)
for i, (c, *_) in enumerate(CLASSES):
    ax.annotate(NAME[c], ((x[2 * i] + x[2 * i + 1]) / 2, 0), xytext=(0, -26),
                textcoords='offset points', ha='center', fontsize=8.5, annotation_clip=False)
ax.set_ylabel('Unmet demand ($10^3$ patient-days)')
# Upper left would sit on top of the antidiabetics bars, so anchor to the
# upper right, which is clear in all three groups.
ax.legend(frameon=False, loc='upper right', bbox_to_anchor=(1.0, 1.0))
#ax.set_title('The total is unchanged; only the split moves', loc='left')
ax.margins(y=0.16)
ax.grid(axis='x', visible=False)

# Right panel: decomposition of the burden
y = np.arange(len(CLASSES))[::-1]
ax = axes[1]
rows = []
for yi, (c, *_) in zip(y, CLASSES):
    h1, o1 = S[c][1.0]; h3, o3 = S[c][3.0]
    burden = 100 * (o3 / o1 - 1)
    ax.barh(yi, burden, height=0.55, color=COL[c], edgecolor='white', linewidth=2)
    ax.text(burden + 2, yi, '%.1f%%' % burden, va='center', fontsize=8)
    rows.append(dict(Group=NAME[c],
                     High_risk_demand_share=('%.1f%%' % (100 * demand_share(c))),
                     h1_o1=round(h1 / o1, 3),
                     Reduction=('%.1f%%' % (100 * (1 - h3 / h1))),
                     Product_h1o1_x_reduction=('%.1f%%' % (100 * (h1 / o1) * (1 - h3 / h1))),
                     Observed_burden=('%.1f%%' % burden)))
ax.set_yticks(y); ax.set_yticklabels([NAME[c] for c, *_ in CLASSES])
ax.set_xlabel('Increase in other-ingredient unmet (%)')
#ax.set_title('Increase in unmet demand', loc='left')
ax.grid(axis='y', visible=False); ax.set_xlim(0, 78)
fig.tight_layout()
savefig(fig, 'fig_sens_rhigh2.png')
plt.show()

print('Decomposition of the burden: (h1/o1) x reduction')
display(pd.DataFrame(rows).set_index('Group'))

### The demand share does not explain it

The explanation "the wider the high-risk designation, the larger the burden" does not
hold across the three groups. Antiepileptics have the smallest high-risk demand share
(17.5%) and yet carry a larger burden than antidiabetics (35.2%).

What determines the burden is not the demand share itself but **how much larger the
high-risk unmet demand is than that of the other ingredients at $r=1$** ($h_1/o_1$).

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.4))
for c, *_ in CLASSES:
    h1, o1 = S[c][1.0]; h3, o3 = S[c][3.0]
    ax.scatter(h1 / o1, 100 * (o3 / o1 - 1), s=110, color=COL[c], marker=MK[c],
               edgecolors='white', linewidths=1.5, zorder=3)
    ax.annotate('%s\ndemand share %.1f%%' % (NAME[c], 100 * demand_share(c)),
                (h1 / o1, 100 * (o3 / o1 - 1)), xytext=(9, -4),
                textcoords='offset points', fontsize=7.5, va='top')
xs = np.linspace(0.2, 1.05, 50)
ax.plot(xs, 100 * xs * 0.762, color=INK2, lw=1.0, ls=(0, (3, 2)), zorder=1)
ax.annotate('theoretical line at a 76.2% reduction', (xs[-1], 100 * xs[-1] * 0.762),
            xytext=(-4, 8), textcoords='offset points', ha='right', fontsize=7.5, color=INK2)
ax.set_xlabel(r'High-risk unmet / other-ingredient unmet at $r=1$ ($h_1/o_1$)')
ax.set_ylabel('Increase in other-ingredient unmet (%)')
ax.set_xlim(0.15, 1.35); ax.set_ylim(0, 78)
plt.show()